In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
from src.imputation import repeated_imputation
from src.missingness import repeated_simulated_missingness
from src.evaluation import get_errors, measure_reconstruction

In [3]:
df = pd.read_csv(PROJECT_ROOT / "data" / "ETTm1.csv", parse_dates=["date"], index_col="date")

# Imputation Pipeline

Use only train data.

### Normalisation

Necessary as imputed values are from different features, so aggregation of reconstruction accuracy between features requires a commonly scaling.

This allows for the use of MAE instead of MAPE.

In [27]:
from sklearn.preprocessing import MinMaxScaler

scaleMinMax = MinMaxScaler(feature_range=(0, 1))

train = scaleMinMax.fit_transform(df.iloc[:1000])

train = pd.DataFrame(train, columns=df.columns).set_index(df.iloc[:1000].index)


### Simulate Missingness with Masks

In [5]:
mechanisms = ["mar", "mcar", "mnar"]
proportions = [10, 30, 50]

missing_dict, mask_dict = repeated_simulated_missingness(df=train, mechanisms=mechanisms, proportions=proportions, iterations=10)

### Imputed With Models

In [16]:
models = ["locf", "ks", "arima", "lstm"]

imputed_data = repeated_imputation(models=models, missing_dict=missing_dict)

### Calculate Errors in Reconstructed Values

In [ ]:
errors = get_errors(df=train, imputed_data=imputed_data, mask_dict=mask_dict, models=models)

### Derive Reconstruction Metrics

In [ ]:
reconstruction = measure_reconstruction(errors=errors, models=models)